# Phase 1-3: 주파수 대역별 파워 분석

**목표**: CEEDNet의 occlusion sensitivity 결과를 PSD 분석으로 직접 확인

**주파수 대역**:
- Delta: 0.5-4 Hz
- Theta: 4-8 Hz
- Alpha: 8-13 Hz
- Beta: 13-30 Hz
- Gamma: 30-45 Hz

**핵심 가설 검증**:
- Dementia: Delta/Theta 증가, Alpha/Beta 감소 (기존 문헌)
- MCI: Normal과 Dementia 사이의 중간 패턴 확인
- MCI 내부 heterogeneity 확인

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal, stats
from collections import defaultdict

sys.path.insert(0, os.path.abspath('..'))

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

DATASET_PATH = '../local/dataset/caueeg-dataset'
SAMPLING_RATE = 200

# 주파수 대역 정의
FREQ_BANDS = {
    'Delta': (0.5, 4),
    'Theta': (4, 8),
    'Alpha': (8, 13),
    'Beta': (13, 30),
    'Gamma': (30, 45),
}

# 채널 정보
CHANNEL_NAMES = ['Fp1', 'F3', 'C3', 'P3', 'O1', 'Fp2', 'F4', 'C4', 'P4', 'O2',
                 'F7', 'T3', 'T5', 'F8', 'T4', 'T6', 'FZ', 'CZ', 'PZ']

BRAIN_REGIONS = {
    'Frontal':  [0, 5, 1, 6, 10, 13, 16],   # Fp1, Fp2, F3, F4, F7, F8, FZ
    'Central':  [2, 7, 17],                   # C3, C4, CZ
    'Parietal': [3, 8, 18],                   # P3, P4, PZ
    'Temporal': [11, 14, 12, 15],             # T3, T4, T5, T6
    'Occipital':[4, 9],                       # O1, O2
}

In [ ]:
from datasets.caueeg_script import load_caueeg_task_datasets

task_config, train_ds, val_ds, test_ds = load_caueeg_task_datasets(
    DATASET_PATH, task='dementia', load_event=True, file_format='edf'
)
class_names = task_config['class_label_to_name']
print(f"Loaded: train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}")

## 1. PSD 계산 유틸리티 및 이벤트 기반 구간 추출

In [ ]:
def compute_band_power(sig, fs, band, nperseg=400):
    """단일 채널 신호의 특정 주파수 대역 파워를 계산"""
    nperseg = min(nperseg, len(sig))
    if nperseg < 4:
        return 0.0
    noverlap = nperseg // 2
    freqs, psd = signal.welch(sig, fs=fs, nperseg=nperseg, noverlap=noverlap)
    idx = np.logical_and(freqs >= band[0], freqs <= band[1])
    if not np.any(idx):
        return 0.0
    return np.trapz(psd[idx], freqs[idx])

def compute_psd(sig, fs, nperseg=400):
    """단일 채널 신호의 PSD를 계산"""
    nperseg = min(nperseg, len(sig))
    if nperseg < 4:
        return np.array([]), np.array([])
    noverlap = nperseg // 2
    return signal.welch(sig, fs=fs, nperseg=nperseg, noverlap=noverlap)

def extract_event_segments(events, signal_length):
    """이벤트 리스트에서 EO/EC 구간을 추출"""
    eo_segments = []
    ec_segments = []
    
    eo_ec_events = [(idx, name) for idx, name in events 
                     if name in ('Eyes Open', 'Eyes Closed')]
    
    for i in range(len(eo_ec_events)):
        start_idx = eo_ec_events[i][0]
        event_name = eo_ec_events[i][1]
        
        if i + 1 < len(eo_ec_events):
            end_idx = eo_ec_events[i + 1][0]
        else:
            end_idx = signal_length
        
        # 최소 2초 (400 samples) 이상인 구간만
        if end_idx - start_idx >= 400:
            if event_name == 'Eyes Open':
                eo_segments.append((start_idx, end_idx))
            else:
                ec_segments.append((start_idx, end_idx))
    
    return eo_segments, ec_segments

print("Utility functions defined.")

## 2. 전체 녹음 PSD: Normal vs MCI vs Dementia

전체 녹음의 PSD를 class별로 비교합니다.

In [ ]:
# 전체 데이터셋에서 band power 계산 (train set만 사용)
print("Computing band powers for all training samples...")

band_power_records = []
for i in range(len(train_ds)):
    sample = train_ds[i]
    eeg = sample['signal'][:19]  # EEG channels only
    class_label = sample['class_label']
    
    # 녹음 중간 60초 구간 사용 (안정적 구간)
    mid = eeg.shape[1] // 2
    start = max(0, mid - 6000)  # 30초 전
    end = min(eeg.shape[1], mid + 6000)  # 30초 후
    eeg_segment = eeg[:, start:end]
    
    # 각 채널, 각 대역의 파워 계산
    for ch_idx in range(19):
        ch_powers = {}
        total_power = 0
        for band_name, band_range in FREQ_BANDS.items():
            power = compute_band_power(eeg_segment[ch_idx], SAMPLING_RATE, band_range)
            ch_powers[band_name] = power
            total_power += power
        
        # 상대 파워 (relative power)
        for band_name in FREQ_BANDS:
            ch_powers[f'{band_name}_rel'] = ch_powers[band_name] / (total_power + 1e-10)
        
        ch_powers['channel'] = CHANNEL_NAMES[ch_idx]
        ch_powers['channel_idx'] = ch_idx
        ch_powers['class_label'] = class_label
        ch_powers['class_name'] = class_names[class_label]
        ch_powers['serial'] = sample['serial']
        ch_powers['alpha_theta_ratio'] = ch_powers['Alpha'] / (ch_powers['Theta'] + 1e-10)
        
        band_power_records.append(ch_powers)
    
    if (i + 1) % 100 == 0:
        print(f"  {i+1}/{len(train_ds)} processed")

df_power = pd.DataFrame(band_power_records)
print(f"Done. Total records: {len(df_power)}")

In [ ]:
# 전체 평균 상대 파워: Class별 비교
colors = {'Normal': '#2ecc71', 'MCI': '#f39c12', 'Dementia': '#e74c3c'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 절대 파워 (log scale)
band_names = list(FREQ_BANDS.keys())
for cls in ['Normal', 'MCI', 'Dementia']:
    means = []
    stds = []
    for band in band_names:
        vals = df_power[df_power['class_name'] == cls].groupby('serial')[band].mean()
        means.append(np.log10(vals.mean() + 1e-10))
        stds.append(np.log10(vals.std() + 1e-10))
    axes[0].plot(band_names, means, 'o-', color=colors[cls], label=cls, linewidth=2, markersize=8)

axes[0].set_ylabel('log10(Mean Absolute Power)')
axes[0].set_title('Absolute Band Power by Class')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 상대 파워
for cls in ['Normal', 'MCI', 'Dementia']:
    means = []
    for band in band_names:
        vals = df_power[df_power['class_name'] == cls].groupby('serial')[f'{band}_rel'].mean()
        means.append(vals.mean())
    axes[1].plot(band_names, means, 'o-', color=colors[cls], label=cls, linewidth=2, markersize=8)

axes[1].set_ylabel('Mean Relative Power')
axes[1].set_title('Relative Band Power by Class')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. 뇌 영역별 상대 파워 비교 (Class별)

In [ ]:
# 영역별 평균 파워 계산
df_power['region'] = df_power['channel_idx'].map(
    {ch_idx: region for region, indices in BRAIN_REGIONS.items() for ch_idx in indices}
)

# 영역별 상대 파워 히트맵
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
region_order = ['Frontal', 'Central', 'Parietal', 'Temporal', 'Occipital']

for ax_idx, cls in enumerate(['Normal', 'MCI', 'Dementia']):
    cls_data = df_power[df_power['class_name'] == cls]
    heatmap_data = cls_data.groupby('region')[[f'{b}_rel' for b in band_names]].mean()
    heatmap_data = heatmap_data.reindex(region_order)
    heatmap_data.columns = band_names
    
    sns.heatmap(heatmap_data, annot=True, fmt='.3f', cmap='YlOrRd', 
                ax=axes[ax_idx], vmin=0, vmax=0.5)
    axes[ax_idx].set_title(f'{cls}', fontweight='bold')
    axes[ax_idx].set_ylabel('Brain Region' if ax_idx == 0 else '')

plt.suptitle('Relative Band Power by Brain Region and Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Alpha/Theta 비율 (핵심 바이오마커)

Alpha/Theta ratio는 치매 진단의 주요 EEG 바이오마커입니다.
- 정상: 높은 Alpha/Theta ratio
- 치매: 낮은 Alpha/Theta ratio (Alpha 감소, Theta 증가)

In [ ]:
# 녹음별 평균 Alpha/Theta ratio
df_at_ratio = df_power.groupby(['serial', 'class_name'])['alpha_theta_ratio'].mean().reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot
sns.boxplot(data=df_at_ratio, x='class_name', y='alpha_theta_ratio',
            order=['Normal', 'MCI', 'Dementia'], palette=colors, ax=axes[0])
axes[0].set_title('Alpha/Theta Ratio by Class')
axes[0].set_ylabel('Alpha/Theta Ratio')
axes[0].set_ylim(0, df_at_ratio['alpha_theta_ratio'].quantile(0.95) * 1.2)

# Violin plot
sns.violinplot(data=df_at_ratio, x='class_name', y='alpha_theta_ratio',
               order=['Normal', 'MCI', 'Dementia'], palette=colors, ax=axes[1], inner='box')
axes[1].set_title('Alpha/Theta Ratio Distribution')
axes[1].set_ylabel('Alpha/Theta Ratio')
axes[1].set_ylim(0, df_at_ratio['alpha_theta_ratio'].quantile(0.95) * 1.2)

plt.tight_layout()
plt.show()

# 통계 검정
print("=== Alpha/Theta Ratio 통계 ===")
for cls in ['Normal', 'MCI', 'Dementia']:
    vals = df_at_ratio[df_at_ratio['class_name'] == cls]['alpha_theta_ratio']
    print(f"  {cls}: mean={vals.mean():.3f}, std={vals.std():.3f}, median={vals.median():.3f}")

# Kruskal-Wallis
normal_at = df_at_ratio[df_at_ratio['class_name'] == 'Normal']['alpha_theta_ratio']
mci_at = df_at_ratio[df_at_ratio['class_name'] == 'MCI']['alpha_theta_ratio']
dem_at = df_at_ratio[df_at_ratio['class_name'] == 'Dementia']['alpha_theta_ratio']

stat, p = stats.kruskal(normal_at, mci_at, dem_at)
print(f"\nKruskal-Wallis: H={stat:.4f}, p={p:.2e}")

# Pairwise tests
pairs = [('Normal', 'MCI', normal_at, mci_at), 
         ('Normal', 'Dementia', normal_at, dem_at), 
         ('MCI', 'Dementia', mci_at, dem_at)]
print("\nPairwise Mann-Whitney U:")
for c1, c2, v1, v2 in pairs:
    u, p = stats.mannwhitneyu(v1, v2, alternative='two-sided')
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    print(f"  {c1} vs {c2}: U={u:.0f}, p={p:.2e} {sig}")

## 5. EO vs EC 구간별 PSD 비교

Eyes-Open과 Eyes-Closed 구간에서의 주파수 특성 차이를 class별로 비교합니다.
특히 Alpha reactivity (EC에서의 Alpha 증가)의 class별 차이를 확인합니다.

In [ ]:
# EO/EC 구간별 band power 계산
print("Computing EO/EC segment band powers...")

eo_ec_records = []
for i in range(len(train_ds)):
    sample = train_ds[i]
    eeg = sample['signal'][:19]
    events = sample.get('event', [])
    
    eo_segs, ec_segs = extract_event_segments(events, eeg.shape[1])
    
    for condition, segments in [('EO', eo_segs), ('EC', ec_segs)]:
        if not segments:
            continue
        
        # 각 구간의 occipital 채널 (O1=4, O2=9) Alpha power 평균
        alpha_powers = []
        theta_powers = []
        for start, end in segments:
            for ch_idx in [4, 9]:  # O1, O2
                alpha_powers.append(compute_band_power(eeg[ch_idx, start:end], SAMPLING_RATE, (8, 13)))
                theta_powers.append(compute_band_power(eeg[ch_idx, start:end], SAMPLING_RATE, (4, 8)))
        
        eo_ec_records.append({
            'serial': sample['serial'],
            'class_name': class_names[sample['class_label']],
            'condition': condition,
            'occ_alpha': np.mean(alpha_powers),
            'occ_theta': np.mean(theta_powers),
            'alpha_theta_ratio': np.mean(alpha_powers) / (np.mean(theta_powers) + 1e-10),
            'n_segments': len(segments),
        })
    
    if (i + 1) % 200 == 0:
        print(f"  {i+1}/{len(train_ds)} processed")

df_eo_ec = pd.DataFrame(eo_ec_records)
print(f"Done. Records: {len(df_eo_ec)}")

# Alpha reactivity: EC Alpha / EO Alpha
df_eo_pivot = df_eo_ec.pivot_table(index=['serial', 'class_name'], columns='condition', 
                                     values='occ_alpha', aggfunc='mean').reset_index()
df_eo_pivot.columns.name = None
df_eo_pivot['alpha_reactivity'] = df_eo_pivot['EC'] / (df_eo_pivot['EO'] + 1e-10)

# 시각화
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# EO vs EC Alpha power
for cls in ['Normal', 'MCI', 'Dementia']:
    subset = df_eo_ec[(df_eo_ec['class_name'] == cls)]
    eo_alpha = subset[subset['condition'] == 'EO']['occ_alpha'].values
    ec_alpha = subset[subset['condition'] == 'EC']['occ_alpha'].values
    axes[0].scatter(eo_alpha[:50], ec_alpha[:50], alpha=0.5, label=cls, color=colors[cls], s=30)

axes[0].plot([0, axes[0].get_xlim()[1]], [0, axes[0].get_xlim()[1]], 'k--', alpha=0.3)
axes[0].set_xlabel('EO Occipital Alpha Power')
axes[0].set_ylabel('EC Occipital Alpha Power')
axes[0].set_title('EO vs EC Alpha Power (Occipital)')
axes[0].legend()

# Alpha reactivity by class
sns.boxplot(data=df_eo_pivot, x='class_name', y='alpha_reactivity',
            order=['Normal', 'MCI', 'Dementia'], palette=colors, ax=axes[1])
axes[1].axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
axes[1].set_title('Alpha Reactivity (EC/EO) by Class')
axes[1].set_ylabel('Alpha Reactivity')
axes[1].set_ylim(0, df_eo_pivot['alpha_reactivity'].quantile(0.95) * 1.2)

# EO vs EC Alpha/Theta ratio
sns.boxplot(data=df_eo_ec, x='class_name', y='alpha_theta_ratio', hue='condition',
            order=['Normal', 'MCI', 'Dementia'], ax=axes[2])
axes[2].set_title('Alpha/Theta Ratio: EO vs EC by Class')
axes[2].set_ylabel('Alpha/Theta Ratio')
axes[2].set_ylim(0, df_eo_ec['alpha_theta_ratio'].quantile(0.95) * 1.2)

plt.tight_layout()
plt.show()

# Alpha reactivity 통계
print("\n=== Alpha Reactivity (EC/EO) 통계 ===")
for cls in ['Normal', 'MCI', 'Dementia']:
    vals = df_eo_pivot[df_eo_pivot['class_name'] == cls]['alpha_reactivity']
    print(f"  {cls}: mean={vals.mean():.3f}, std={vals.std():.3f}")

## 6. 요약

**확인 사항**:
- [ ] Dementia에서 Delta/Theta 증가, Alpha/Beta 감소 확인 → 기존 문헌과 일치
- [ ] MCI가 Normal과 Dementia 사이 중간 패턴인지 확인
- [ ] Alpha/Theta ratio의 class별 유의미한 차이 확인
- [ ] Alpha reactivity (EC/EO)의 class별 차이 확인
- [ ] 뇌 영역별로 가장 큰 차이를 보이는 영역 확인 → GCN 설계 근거

**다음 단계**: Phase 1-4에서 전환 시점 전후의 동적 패턴을 시간-주파수 분석으로 상세히 확인합니다.